# Statistical analysis

This notebook performs EDA and statistical tests comparing device conversion rates (mobile vs desktop).

Instructions (fill paths after exporting SQL results):
1. Export `funnel_by_device` or `user_funnel` from BigQuery to `data/processed/funnel_by_device.csv` and/or `data/processed/user_funnel.csv`.
2. Update the file paths in the data-loading cell below.

NOTE: This notebook contains placeholders. Do NOT claim results until you run the queries and populate the CSVs.

In [1]:


from pathlib import Path

import pandas as pd
import numpy as np
from scipy.stats import norm
from statsmodels.stats.proportion import proportions_ztest

possible_paths = [
    Path("data/processed/demo/device_funnel.csv"),
    Path("../data/processed/demo/device_funnel.csv"),
]

csv_path = next(path for path in possible_paths if path.exists())
device_funnel = pd.read_csv(csv_path)

device_funnel

,device_category,product_view_sessions,checkout_sessions,purchase_sessions,view_to_checkout_rate,checkout_to_purchase_rate,overall_purchase_rate
0,mobile,30501,4329,1910,14.19,44.12,6.26
1,desktop,44819,6206,2651,13.85,42.72,5.91
2,tablet,1700,235,100,13.82,42.55,5.88


In [2]:
mobile = device_funnel.loc[
    device_funnel["device_category"] == "mobile"
].iloc[0]

desktop = device_funnel.loc[
    device_funnel["device_category"] == "desktop"
].iloc[0]

successes = np.array([
    mobile["purchase_sessions"],
    desktop["purchase_sessions"],
])

observations = np.array([
    mobile["product_view_sessions"],
    desktop["product_view_sessions"],
])

z_statistic, p_value = proportions_ztest(
    count=successes,
    nobs=observations,
    alternative="two-sided",
)

mobile_rate = successes[0] / observations[0]
desktop_rate = successes[1] / observations[1]
rate_difference = mobile_rate - desktop_rate

standard_error = np.sqrt(
    mobile_rate * (1 - mobile_rate) / observations[0]
    + desktop_rate * (1 - desktop_rate) / observations[1]
)

critical_value = norm.ppf(0.975)

ci_lower = rate_difference - critical_value * standard_error
ci_upper = rate_difference + critical_value * standard_error

print(f"Mobile conversion rate: {mobile_rate:.4%}")
print(f"Desktop conversion rate: {desktop_rate:.4%}")
print(f"Difference: {rate_difference:.4%}")
print(f"Z-statistic: {z_statistic:.4f}")
print(f"P-value: {p_value:.4f}")
print(
    "95% CI for difference: "
    f"({ci_lower:.4%}, {ci_upper:.4%})"
)

Mobile conversion rate: 6.2621%
Desktop conversion rate: 5.9149%
Difference: 0.3472%
Z-statistic: 1.9610
P-value: 0.0499
95% CI for difference: (-0.0016%, 0.6959%)


## Mobile vs. Desktop Conversion

A two-proportion z-test compared purchase conversion among mobile and
desktop product-view sessions.

- Mobile: 1,910 purchases across 30,501 sessions (6.26%)
- Desktop: 2,651 purchases across 44,819 sessions (5.91%)
- Observed difference: 0.35 percentage points
- Z-statistic: 1.961
- P-value: 0.0499
- 95% Wald confidence interval: −0.002 to 0.696 percentage points

The results provide only borderline evidence of a difference in conversion
between mobile and desktop sessions. Although the pooled z-test produced a
p-value just below 0.05, the confidence interval was effectively centered on
the significance boundary and included zero by a very small amount.

The estimated effect was also modest: mobile conversion was approximately
0.35 percentage points higher than desktop conversion. This observational
comparison does not establish that device type caused the difference.
Traffic source, customer characteristics, product selection, and other
variables may confound the relationship.

The practical conclusion is that the data does not support the assumption
that mobile sessions perform worse than desktop sessions. Additional data
or a controlled experiment would be needed before recommending a
device-specific intervention.